# Z-Image Turbo on Kaggle (ComfyUI)

Z-Image Turbo (Tongyi-MAI / Alibaba, 6B S3-DiT, Apache 2.0, commercial OK). ComfyUI native support.
Official template: Comfy-Org/workflow_templates `image_z_image_turbo.json` (subgraph expanded below).
Nodes: UNETLoader -> ModelSamplingAuraFlow (shift 3) -> KSampler (8 steps, cfg 1, res_multistep, simple);
CLIPLoader type lumina2 (qwen_3_4b) -> CLIPTextEncode -> ConditioningZeroOut (negative);
EmptySD3LatentImage; VAEDecode; SaveImage.

Files (sizes confirmed via HF API, Comfy-Org/Z-Image-Turbo split_files/):
- diffusion z_image_turbo_int8_convrot.safetensors 6.20 GB (INT8; BF16 12.31 GB + BF16 encoder 8.04 GB + VAE 0.34 GB = 20.69 GB would exceed the 20 GB /kaggle/working limit)
- text encoder qwen_3_4b_fp8_mixed.safetensors 5.63 GB (FP8; BF16 encoder is 8.04 GB)
- VAE ae.safetensors 0.34 GB (Flux VAE)
Total 12.17 GB, fits /kaggle/working 20 GB. T4 16 GB: FP8/INT8 route runs comfortably.

LoRA (optional, reserved for later use): same conditional LoraLoaderModelOnly pattern as the MOSS/Qwen/Krea2 notebooks.
Set LORA_NAME to a .safetensors file in ComfyUI/models/loras/ and LORA_REPO to its HF repo (file at repo root).
Leave LORA_NAME = "" (default) to run without a LoRA. Z-Image LoRAs must be trained for Z-Image; FLUX/Qwen LoRAs will NOT work.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
!if [ -d /kaggle/working/ComfyUI ]; then echo comfyui-exists; else git clone --depth 1 https://github.com/comfyanonymous/ComfyUI.git /kaggle/working/ComfyUI; fi
!pip install -q -r /kaggle/working/ComfyUI/requirements.txt

In [ ]:
import os, shutil
os.environ["HF_HOME"] = "/kaggle/working/hf_cache"
from huggingface_hub import hf_hub_download
base = "/kaggle/working/ComfyUI/models"
#
# LoRA (optional, reserved for later use): set LORA_NAME to a .safetensors file
# in ComfyUI/models/loras/ and LORA_REPO to the Hugging Face repo hosting it
# (file assumed at the repo root). Leave LORA_NAME = "" (default) to run
# without a LoRA. When set, the file is downloaded here and the run cell wires
# a LoraLoaderModelOnly node into the model line.
LORA_NAME = ""
LORA_REPO = ""
LORA_STRENGTH = 1.0
# Z-Image Turbo files (sizes confirmed via HF API, Comfy-Org/Z-Image-Turbo split_files/):
# - diffusion z_image_turbo_int8_convrot.safetensors 6.20 GB (official Comfy-Org INT8 repack)
# - text encoder qwen_3_4b_fp8_mixed.safetensors 5.63 GB (official Comfy-Org FP8 repack)
# - VAE ae.safetensors 0.34 GB (Flux VAE)
# Total ~12.17 GB, fits /kaggle/working 20 GB.
# NOTE: hf_hub_download mirrors the repo subdirectory (split_files/...) under local_dir,
# so each file is moved to the folder its loader expects (the Wan V3/V4 lesson).
def dl(repo, fname, dest_folder, dest_name):
    p = hf_hub_download(repo_id=repo, filename=fname, local_dir=base)
    target = os.path.join(base, dest_folder, dest_name)
    os.makedirs(os.path.dirname(target), exist_ok=True)
    if os.path.abspath(p) != os.path.abspath(target):
        shutil.move(p, target)
    print(dest_folder + "/" + dest_name, round(os.path.getsize(target) / 1e9, 2), "GB")
    return target

dl("Comfy-Org/Z-Image-Turbo", "split_files/diffusion_models/z_image_turbo_int8_convrot.safetensors", "diffusion_models", "z_image_turbo_int8_convrot.safetensors")
dl("Comfy-Org/Z-Image-Turbo", "split_files/text_encoders/qwen_3_4b_fp8_mixed.safetensors", "text_encoders", "qwen_3_4b_fp8_mixed.safetensors")
dl("Comfy-Org/Z-Image-Turbo", "split_files/vae/ae.safetensors", "vae", "ae.safetensors")
if LORA_NAME:
    p = hf_hub_download(repo_id=LORA_REPO, filename=LORA_NAME, local_dir=base + "/loras")
    print("lora", LORA_NAME, round(os.path.getsize(p) / 1e9, 2), "GB")
# Guard: the three files must sit exactly where the loaders look, or fail now.
expected = {
    "diffusion_models": "z_image_turbo_int8_convrot.safetensors",
    "text_encoders": "qwen_3_4b_fp8_mixed.safetensors",
    "vae": "ae.safetensors",
}
for folder, fname in expected.items():
    fp = os.path.join(base, folder, fname)
    print(folder + "/", fname, "exists:", os.path.exists(fp))
    assert os.path.exists(fp), "missing model file: " + fp
print("all models downloaded")

In [ ]:
import json, time, requests, os, subprocess, sys

BASE = "http://127.0.0.1:8188"
LOG = "/kaggle/working/comfy_server.log"

# Start the ComfyUI server as a child of this cell's python process (nohup across cells does not survive Kaggle version runs).
logf = open(LOG, "wb")
proc = subprocess.Popen(
    [sys.executable, "main.py", "--listen", "127.0.0.1", "--port", "8188", "--disable-auto-launch"],
    cwd="/kaggle/working/ComfyUI",
    stdout=logf, stderr=subprocess.STDOUT,
    start_new_session=True,
)
print("server pid:", proc.pid)

up = False
for i in range(36):
    if proc.poll() is not None:
        print("server exited early, code:", proc.returncode)
        break
    try:
        r = requests.get(BASE + "/system_stats", timeout=5)
        if r.status_code == 200:
            up = True
            print("server up after", (i + 1) * 5, "s")
            break
    except Exception:
        pass
    time.sleep(5)
if not up:
    print("server did not come up; log tail:")
    print(open(LOG, errors="ignore").read()[-3000:])
    proc.terminate()
    raise SystemExit("comfyui server failed to start")

PROMPT = "A photorealistic portrait of a young woman in a sunlit garden, soft morning light, shallow depth of field, 50mm lens"
WIDTH = 1024
HEIGHT = 1024
# LoRA settings are defined in the download cell; default to "no LoRA" if that cell was skipped.
LORA_NAME = globals().get("LORA_NAME", "")
LORA_STRENGTH = globals().get("LORA_STRENGTH", 1.0)

info = requests.get(BASE + "/object_info", timeout=120).json()
have = set(info.keys())
need = ["UNETLoader", "CLIPLoader", "VAELoader", "CLIPTextEncode", "ConditioningZeroOut",
        "EmptySD3LatentImage", "ModelSamplingAuraFlow", "KSampler", "VAEDecode", "SaveImage"]
if LORA_NAME:
    need.append("LoraLoaderModelOnly")
missing = [n for n in need if n not in have]
print("missing nodes:", missing if missing else "none")
if missing:
    proc.terminate()
    raise SystemExit("missing nodes: " + ", ".join(missing))

def node_inputs(class_type, preferred):
    spec = info[class_type]["input"]
    allowed = set(spec.get("required", {})) | set(spec.get("optional", { }))
    return {k: v for k, v in preferred.items() if k in allowed}

# Print the loader + sampler + clip specs to the log so a wrong input name/value is caught here, not after a long run.
for ct in ["UNETLoader", "CLIPLoader", "ModelSamplingAuraFlow", "KSampler", "EmptySD3LatentImage"]:
    print(ct, "spec:", json.dumps(info[ct]["input"])[:1200])

# Raw-node version of the official Z-Image Turbo T2I template (subgraph expanded,
# prompt-enhancement branch dropped, prompt sent as-is). Core pipeline:
# UNETLoader -> ModelSamplingAuraFlow (shift 3) -> KSampler 8 steps, cfg 1,
# res_multistep, simple; CLIPLoader type lumina2 -> CLIPTextEncode ->
# ConditioningZeroOut (negative); EmptySD3LatentImage 1024x1024; VAEDecode; SaveImage.
nodes = {}
nodes["1"] = {"class_type": "UNETLoader", "inputs": node_inputs("UNETLoader", {"unet_name": "z_image_turbo_int8_convrot.safetensors", "weight_dtype": "default"})}
nodes["2"] = {"class_type": "CLIPLoader", "inputs": node_inputs("CLIPLoader", {"clip_name": "qwen_3_4b_fp8_mixed.safetensors", "type": "lumina2", "device": "default"})}
nodes["3"] = {"class_type": "VAELoader", "inputs": node_inputs("VAELoader", {"vae_name": "ae.safetensors"})}
nodes["4"] = {"class_type": "CLIPTextEncode", "inputs": {"text": PROMPT, "clip": ["2", 0]}}
nodes["5"] = {"class_type": "ConditioningZeroOut", "inputs": {"conditioning": ["4", 0]}}
nodes["6"] = {"class_type": "EmptySD3LatentImage", "inputs": node_inputs("EmptySD3LatentImage", {"width": WIDTH, "height": HEIGHT, "batch_size": 1})}
# Model line: UNETLoader -> (optional LoRA) -> ModelSamplingAuraFlow (shift 3) -> KSampler.
# LoRA (optional): when LORA_NAME is set, patch the model with LoraLoaderModelOnly
# (model-only, no CLIP) between the UNET loader and ModelSamplingAuraFlow. When empty,
# the model goes straight from the UNET loader to ModelSamplingAuraFlow, unchanged.
MODEL_OUT = ["1", 0]
if LORA_NAME:
    nodes["20"] = {"class_type": "LoraLoaderModelOnly", "inputs": node_inputs("LoraLoaderModelOnly", {"model": ["1", 0], "lora_name": LORA_NAME, "strength_model": LORA_STRENGTH})}
    MODEL_OUT = ["20", 0]
nodes["7"] = {"class_type": "ModelSamplingAuraFlow", "inputs": node_inputs("ModelSamplingAuraFlow", {"model": MODEL_OUT, "shift": 3})}
nodes["8"] = {"class_type": "KSampler", "inputs": {"model": ["7", 0], "positive": ["4", 0], "negative": ["5", 0], "latent_image": ["6", 0], "seed": 42, "steps": 8, "cfg": 1, "sampler_name": "res_multistep", "scheduler": "simple", "denoise": 1.0}}
nodes["9"] = {"class_type": "VAEDecode", "inputs": {"samples": ["8", 0], "vae": ["3", 0]}}
nodes["10"] = {"class_type": "SaveImage", "inputs": {"images": ["9", 0], "filename_prefix": "zimage_test"}}

# Pre-check: every required input of every node must be present, or fail here with the
# full list (do not let the server reject it one input at a time). Plus a combo-value
# check: where the node definition lists the allowed values, the value we send must be
# one of them (a bad value can be silently dropped and only fail at execution -- the Wan
# V5 lesson). NOTE: a combo's allowed values live in cfg[0] as a LIST; a scalar input
# (STRING/INT/FLOAT) has cfg[0] as a type-name string, so it is skipped -- this is the
# exact check the Qwen/Krea2 notebooks use.
problems = []
for nid, node in nodes.items():
    spec = info[node["class_type"]]["input"]
    req = spec.get("required", {})
    for name in req:
        if name not in node["inputs"]:
            problems.append(nid + ":" + node["class_type"] + "." + name)
    for name, val in node["inputs"].items():
        cfg = req.get(name) or spec.get("optional", {}).get(name)
        if isinstance(cfg, list) and cfg and isinstance(cfg[0], list) and isinstance(val, (str, int, float)):
            if val not in cfg[0]:
                problems.append(nid + ":" + node["class_type"] + "." + name + " value '" + str(val) + "' not in " + str(cfg[0]))
print("missing required inputs:", problems if problems else "none")
if problems:
    proc.terminate()
    raise SystemExit("missing required inputs: " + ", ".join(problems))

r = requests.post(BASE + "/prompt", json={"prompt": nodes}, timeout=180)
print("submit:", r.status_code, r.text[:400])
r.raise_for_status()
pid = r.json()["prompt_id"]
print("prompt_id:", pid)

out_path = "/kaggle/working/zimage_test.png"
t0 = time.time()
done = False
# NOTE: SaveImage writes into ComfyUI/output/, NOT /kaggle/working/. The image must
# be fetched back through the /view API and written to out_path here -- the Krea2 V2
# failure was exactly this step missing (sampling finished and node 10 had produced
# zimage_test_00001_.png, but nothing copied it to out_path, so the assert below
# failed). This is the same download step the Qwen/MOSS/Krea2 notebooks use.
while time.time() - t0 < 3600:
    h = requests.get(BASE + "/history/" + pid, timeout=60).json()
    if pid in h:
        st = h[pid].get("status", {})
        sstr = st.get("status_str")
        if sstr == "success":
            outs = h[pid]["outputs"]
            imgs = outs.get("10", {}).get("images", [])
            print("images:", imgs)
            if imgs:
                im = imgs[0]
                q = {"filename": im["filename"], "subfolder": im.get("subfolder", ""), "type": im.get("type", "output")}
                data = requests.get(BASE + "/view", params=q, timeout=300).content
                open(out_path, "wb").write(data)
                print("saved", out_path, len(data), "bytes")
                done = True
        break
    time.sleep(30)
print("done:", done, "elapsed:", round(time.time() - t0, 1), "s")
proc.terminate()
print("server stopped")
print("exists:", os.path.exists(out_path), "bytes:", os.path.getsize(out_path) if os.path.exists(out_path) else 0)
assert os.path.exists(out_path), "no output image: " + out_path


In [ ]:
from IPython.display import Image
Image("/kaggle/working/zimage_test.png")